In [1]:
import numpy as np  # linear algebra
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-whitegrid")



/tmp/ipykernel_11/3727763869.py:6: MatplotlibDeprecationWarning: The seaborn styles shipped by Matplotlib are deprecated since 3.6, as they no longer correspond to the styles shipped by seaborn. However, they will remain available as 'seaborn-v0_8-<style>'. Alternatively, directly use the seaborn API instead.
  plt.style.use("seaborn-whitegrid")


In [2]:
df_train = pd.read_csv(
    "../input/new-york-city-taxi-fare-prediction/train.csv",
    nrows=1_000_000,
    parse_dates=["pickup_datetime"],
)
df_train.head()



,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,2009-06-15 17:26:21.0000001,4.5,2009-06-15 17:26:21+00:00,-73.844311,40.721319,-73.841610,40.712278,1
1,2010-01-05 16:52:16.0000002,16.9,2010-01-05 16:52:16+00:00,-74.016048,40.711303,-73.979268,40.782004,1
2,2011-08-18 00:35:00.00000049,5.7,2011-08-18 00:35:00+00:00,-73.982738,40.761270,-73.991242,40.750562,2
3,2012-04-21 04:30:42.0000001,7.7,2012-04-21 04:30:42+00:00,-73.987130,40.733143,-73.991567,40.758092,1
4,2010-03-09 07:51:00.000000135,5.3,2010-03-09 07:51:00+00:00,-73.968095,40.768008,-73.956655,40.783762,1


In [3]:
df_train.describe()



,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,1000000.000000,1000000.000000,1000000.000000,999990.000000,999990.000000,1000000.000000
mean,11.348079,-72.526640,39.929008,-72.527860,39.919954,1.684924
std,9.822090,12.057937,7.626154,11.324494,8.201418,1.323911
min,-44.900000,-3377.680935,-3116.285383,-3383.296608,-3114.338567,0.000000
25%,6.000000,-73.992060,40.734965,-73.991385,40.734046,1.000000
50%,8.500000,-73.981792,40.752695,-73.980135,40.753166,1.000000
75%,12.500000,-73.967094,40.767154,-73.963654,40.768129,2.000000
max,500.000000,2522.271325,2621.628430,45.581619,1651.553433,208.000000


In [4]:
df_test = pd.read_csv(
    "../input/new-york-city-taxi-fare-prediction/test.csv",
    parse_dates=["pickup_datetime"],
)
df_test.head()



,key,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,2010-10-01 21:26:11.0000001,2010-10-01 21:26:11+00:00,-73.983130,40.761970,-73.994386,40.749236,1
1,2013-10-06 01:38:00.00000083,2013-10-06 01:38:00+00:00,-73.948505,40.753977,-73.808195,40.731952,2
2,2012-03-30 19:13:53.0000001,2012-03-30 19:13:53+00:00,-73.973964,40.791979,-73.979018,40.785544,1
3,2012-02-08 02:57:23.0000001,2012-02-08 02:57:23+00:00,-73.991478,40.738907,-73.907198,40.861572,2
4,2013-12-13 22:56:00.000000237,2013-12-13 22:56:00+00:00,-73.986281,40.740067,-73.933927,40.856781,2


In [5]:
df_test.describe()



,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,9914.000000,9914.000000,9914.000000,9914.000000,9914.000000
mean,-72.365437,39.854075,-72.355298,39.849902,1.701735
std,10.901230,6.325354,10.928248,6.338538,1.328327
min,-74.178068,-74.000425,-75.437730,-74.009507,0.000000
25%,-73.992274,40.734688,-73.991365,40.733664,1.000000
50%,-73.981978,40.752504,-73.980220,40.753209,1.000000
75%,-73.966718,40.767114,-73.963598,40.768143,2.000000
max,40.790297,41.366138,40.800715,41.056380,6.000000


In [6]:
print("Old size: %d" % len(df_train))
df_train = df_train[df_train.fare_amount >= 0]
print("New size: %d" % len(df_train))



Old size: 1000000
New size: 999962


In [7]:
print("Old size: %d" % len(df_train))
df_train = df_train.dropna(how="any", axis="rows")
print("New size: %d" % len(df_train))



Old size: 999962
New size: 999952


In [8]:
df_train[df_train.fare_amount < 80].fare_amount.hist(bins=100)
plt.xlabel("fare $USD")
plt.close()



In [9]:
df_train["diff_long"] = (df_train.dropoff_longitude - df_train.pickup_longitude).abs()
df_train["diff_long"].describe()



count    999952.000000
mean          0.182548
std           7.676120
min           0.000000
25%           0.005803
50%           0.012447
75%           0.023699
max        4240.388978
Name: diff_long, dtype: float64

In [10]:
df_train["diff_lat"] = (df_train.dropoff_latitude - df_train.pickup_latitude).abs()
df_train["diff_lat"].describe()



count    999952.000000
mean          0.113383
std           8.270898
min           0.000000
25%           0.006592
50%           0.013865
75%           0.026917
max        5486.099433
Name: diff_lat, dtype: float64

In [11]:
print("Old size: %d" % len(df_train))
df_train = df_train[(df_train.diff_long < 5.0) & (df_train.diff_lat < 5.0)]
print("New size: %d" % len(df_train))



Old size: 999952
New size: 997990


In [12]:
df_train["year"] = df_train["pickup_datetime"].dt.year
df_train["weekday"] = df_train["pickup_datetime"].dt.weekday
df_train["hour"] = df_train["pickup_datetime"].dt.hour

df_train["month"] = df_train["pickup_datetime"].dt.month
df_train["dayofmonth"] = df_train["pickup_datetime"].dt.day



In [13]:
df_train.describe()



,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,diff_long,diff_lat,year,weekday,hour,month,dayofmonth
count,997990.000000,997990.000000,997990.000000,997990.000000,997990.000000,997990.000000,997990.000000,997990.000000,997990.000000,997990.000000,997990.000000,997990.000000,997990.000000
mean,11.341837,-72.583098,39.959231,-72.582209,39.959569,1.684950,0.022506,0.021086,2011.739095,3.039942,13.509711,6.267971,15.697861
std,9.801924,10.166602,5.984613,10.166486,5.984640,1.323821,0.038233,0.027717,1.860760,1.949923,6.513913,3.436472,8.682267
min,0.000000,-121.391210,-74.016587,-121.391248,-74.035197,0.000000,0.000000,0.000000,2009.000000,0.000000,0.000000,1.000000,1.000000
25%,6.000000,-73.992070,40.735032,-73.991394,40.734107,1.000000,0.005792,0.006580,2010.000000,1.000000,9.000000,3.000000,8.000000
50%,8.500000,-73.981810,40.752727,-73.980158,40.753195,1.000000,0.012420,0.013832,2012.000000,3.000000,14.000000,6.000000,16.000000
75%,12.500000,-73.967157,40.767174,-73.963734,40.768146,2.000000,0.023603,0.026808,2013.000000,5.000000,19.000000,9.000000,23.000000
max,500.000000,40.850357,51.084373,40.851027,51.346519,208.000000,4.923874,3.820613,2015.000000,6.000000,23.000000,12.000000,31.000000


In [14]:
df_train[["fare_amount", "hour"]].groupby(["hour"], as_index=False).mean().sort_values(
    by="fare_amount", ascending=False
)



,hour,fare_amount
5,5,15.389197
4,4,13.563618
6,6,12.203498
3,3,12.026388
15,15,12.014656
14,14,11.864828
16,16,11.847752
0,0,11.730199
23,23,11.607433
13,13,11.602990


In [15]:
df_train[["fare_amount", "weekday"]].groupby(
    ["weekday"], as_index=False
).mean().sort_values(by="fare_amount", ascending=False)



,weekday,fare_amount
6,6,11.632518
3,3,11.501779
4,4,11.387728
0,0,11.385339
2,2,11.280719
1,1,11.216120
5,5,11.022402


In [16]:
df_train[["fare_amount", "year"]].groupby(["year"], as_index=False).mean().sort_values(
    by="fare_amount", ascending=False
)




,year,fare_amount
6,2015,13.016302
5,2014,12.910192
4,2013,12.606738
3,2012,11.175261
2,2011,10.433560
1,2010,10.177292
0,2009,10.041246


In [17]:
def distance(lat1, lon1, lat2, lon2):
    p = 0.017453292519943295  # Pi/180
    a = (
        0.5
        - np.cos((lat2 - lat1) * p) / 2
        + np.cos(lat1 * p) * np.cos(lat2 * p) * (1 - np.cos((lon2 - lon1) * p)) / 2
    )
    return 12742 * np.arcsin(np.sqrt(a))  # 2*R*asin...




In [18]:
df_train["distance"] = distance(
    df_train.pickup_latitude,
    df_train.pickup_longitude,
    df_train.dropoff_latitude,
    df_train.dropoff_longitude,
)



In [19]:
plot = df_train.plot.scatter("distance", "fare_amount")
plt.close()



In [20]:
plot = df_train[(df_train.distance < 50) & (df_train.fare_amount < 100)].plot.scatter(
    "distance", "fare_amount", alpha=0.1
)
plt.close()



In [21]:
print("Old size: %d" % len(df_train))
df_train = df_train[(df_train.distance >= 0.1)]
print("New size: %d" % len(df_train))



Old size: 997990
New size: 963705


In [22]:
plot = df_train[(df_train.distance < 50) & (df_train.fare_amount < 100)].plot.scatter(
    "distance", "fare_amount", alpha=0.1
)
plt.close()



In [23]:
print("Old size: %d" % len(df_train))
df_train = df_train[(df_train.distance <= 50)]
print("New size: %d" % len(df_train))



Old size: 963705
New size: 963341


In [24]:
plot = df_train.plot.scatter("distance", "fare_amount", alpha=0.1)
plt.close()



In [25]:
print("Old size: %d" % len(df_train))
df_train = df_train[(df_train.fare_amount <= 200)]
print("New size: %d" % len(df_train))



Old size: 963341
New size: 963336


In [26]:
plot = df_train.plot.scatter("distance", "fare_amount", alpha=0.1)
plt.close()



In [27]:
print("Old size: %d" % len(df_train))
df_train = df_train[
    (df_train.pickup_longitude.between(-75.0, -72.0))
    & (df_train.dropoff_longitude.between(-75.0, -72.0))
    & (df_train.pickup_latitude.between(40.0, 42.0))
    & (df_train.dropoff_latitude.between(40.0, 42.0))
]
print("New size: %d" % len(df_train))

for col in ["passenger_count"]:
    df_train[col] = pd.to_numeric(df_train[col], errors="coerce")
df_train = df_train.dropna(subset=["passenger_count"])
df_train = df_train[df_train["passenger_count"].between(1, 6)]
df_train["passenger_count"] = df_train["passenger_count"].astype(np.int16)

print("Old size (before min-fare filter): %d" % len(df_train))
df_train = df_train[df_train["fare_amount"] >= 2.5]
print("New size (after min-fare filter): %d" % len(df_train))

df_train["pickup_longitude"] = pd.to_numeric(
    df_train["pickup_longitude"], errors="coerce"
)
df_train["dropoff_longitude"] = pd.to_numeric(
    df_train["dropoff_longitude"], errors="coerce"
)
df_train["pickup_latitude"] = pd.to_numeric(
    df_train["pickup_latitude"], errors="coerce"
)
df_train["dropoff_latitude"] = pd.to_numeric(
    df_train["dropoff_latitude"], errors="coerce"
)
df_train = df_train.dropna(
    subset=[
        "pickup_longitude",
        "dropoff_longitude",
        "pickup_latitude",
        "dropoff_latitude",
    ]
)

# Change (score-relevant, minimal): clip train coordinates to the same bounds as test
# BEFORE computing final engineered features, to reduce train/test distribution mismatch.
df_train["pickup_longitude"] = df_train["pickup_longitude"].clip(-75.0, -72.0)
df_train["dropoff_longitude"] = df_train["dropoff_longitude"].clip(-75.0, -72.0)
df_train["pickup_latitude"] = df_train["pickup_latitude"].clip(40.0, 42.0)
df_train["dropoff_latitude"] = df_train["dropoff_latitude"].clip(40.0, 42.0)

df_train["diff_long"] = (
    df_train["dropoff_longitude"] - df_train["pickup_longitude"]
).abs()
df_train["diff_lat"] = (
    df_train["dropoff_latitude"] - df_train["pickup_latitude"]
).abs()
df_train["distance"] = distance(
    df_train["pickup_latitude"],
    df_train["pickup_longitude"],
    df_train["dropoff_latitude"],
    df_train["dropoff_longitude"],
)

print("Old size (before GPS-glitch filter): %d" % len(df_train))
glitch_mask = (
    (df_train["diff_long"] < 1e-4)
    & (df_train["diff_lat"] < 1e-4)
    & (df_train["distance"] > 0.2)
)
df_train = df_train[~glitch_mask]
print("New size (after GPS-glitch filter): %d" % len(df_train))

fare_per_km = df_train["fare_amount"] / (df_train["distance"] + 1e-6)
print("Old size (before fare/km filter): %d" % len(df_train))
df_train = df_train[fare_per_km.between(0.5, 60.0)]
print("New size (after fare/km filter): %d" % len(df_train))

fare_per_km = df_train["fare_amount"] / (df_train["distance"] + 1e-6)
print("Old size (before tighter extreme fare/km filter): %d" % len(df_train))
df_train = df_train[fare_per_km <= 22.0]
print("New size (after tighter extreme fare/km filter): %d" % len(df_train))

print("Old size (before long-trip filter): %d" % len(df_train))
df_train = df_train[df_train["distance"] <= 35.0]
print("New size (after long-trip filter): %d" % len(df_train))

print("Old size (before zero-distance nonzero-fare filter): %d" % len(df_train))
df_train = df_train[~((df_train["distance"] < 0.05) & (df_train["fare_amount"] > 10.0))]
print("New size (after zero-distance nonzero-fare filter): %d" % len(df_train))

df_train["abs_lat_diff"] = (
    df_train["dropoff_latitude"] - df_train["pickup_latitude"]
).abs()
df_train["abs_lon_diff"] = (
    df_train["dropoff_longitude"] - df_train["pickup_longitude"]
).abs()

NYC_LAT, NYC_LON = 40.7141667, -74.0063889
df_train["pickup_dist_nyc"] = distance(
    df_train["pickup_latitude"], df_train["pickup_longitude"], NYC_LAT, NYC_LON
)
df_train["dropoff_dist_nyc"] = distance(
    df_train["dropoff_latitude"], df_train["dropoff_longitude"], NYC_LAT, NYC_LON
)

df_train["manhattan_dist"] = df_train["abs_lat_diff"] + df_train["abs_lon_diff"]
df_train["log_distance"] = np.log1p(df_train["distance"])

df_train["bearing"] = np.arctan2(
    (df_train["dropoff_longitude"] - df_train["pickup_longitude"]),
    (df_train["dropoff_latitude"] - df_train["pickup_latitude"]),
).astype(np.float32)

df_train["center_dist_change"] = (
    df_train["dropoff_dist_nyc"] - df_train["pickup_dist_nyc"]
).astype(np.float32)

df_train["pickup_grid_lat"] = np.floor(df_train["pickup_latitude"] * 100.0).astype(
    np.int32
)
df_train["pickup_grid_lon"] = np.floor(df_train["pickup_longitude"] * 100.0).astype(
    np.int32
)
df_train["dropoff_grid_lat"] = np.floor(df_train["dropoff_latitude"] * 100.0).astype(
    np.int32
)
df_train["dropoff_grid_lon"] = np.floor(df_train["dropoff_longitude"] * 100.0).astype(
    np.int32
)

df_train["pickup_latitude"] = df_train["pickup_latitude"].astype(np.float32)
df_train["pickup_longitude"] = df_train["pickup_longitude"].astype(np.float32)
df_train["dropoff_latitude"] = df_train["dropoff_latitude"].astype(np.float32)
df_train["dropoff_longitude"] = df_train["dropoff_longitude"].astype(np.float32)

df_train["year"] = df_train["pickup_datetime"].dt.year
df_train["weekday"] = df_train["pickup_datetime"].dt.weekday
df_train["hour"] = df_train["pickup_datetime"].dt.hour
df_train["month"] = df_train["pickup_datetime"].dt.month
df_train["dayofmonth"] = df_train["pickup_datetime"].dt.day

features = [
    "year",
    "month",
    "dayofmonth",
    "weekday",
    "hour",
    "distance",
    "log_distance",
    "passenger_count",
    "pickup_latitude",
    "pickup_longitude",
    "dropoff_latitude",
    "dropoff_longitude",
    "abs_lat_diff",
    "abs_lon_diff",
    "manhattan_dist",
    "pickup_dist_nyc",
    "dropoff_dist_nyc",
    "bearing",
    "center_dist_change",
    "pickup_grid_lat",
    "pickup_grid_lon",
    "dropoff_grid_lat",
    "dropoff_grid_lon",
]



Old size: 963336
New size: 962683


/tmp/ipykernel_11/305134347.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train[col] = pd.to_numeric(df_train[col], errors="coerce")


Old size (before min-fare filter): 959231
New size (after min-fare filter): 959217
Old size (before GPS-glitch filter): 959217
New size (after GPS-glitch filter): 959217
Old size (before fare/km filter): 959217
New size (after fare/km filter): 957563
Old size (before tighter extreme fare/km filter): 957563
New size (after tighter extreme fare/km filter): 953809
Old size (before long-trip filter): 953809
New size (after long-trip filter): 953738
Old size (before zero-distance nonzero-fare filter): 953738
New size (after zero-distance nonzero-fare filter): 953738


In [28]:
df_test["year"] = df_test["pickup_datetime"].dt.year
df_test["weekday"] = df_test["pickup_datetime"].dt.weekday
df_test["hour"] = df_test["pickup_datetime"].dt.hour

df_test["month"] = df_test["pickup_datetime"].dt.month
df_test["dayofmonth"] = df_test["pickup_datetime"].dt.day

for col in [
    "pickup_latitude",
    "pickup_longitude",
    "dropoff_latitude",
    "dropoff_longitude",
]:
    df_test[col] = pd.to_numeric(df_test[col], errors="coerce")

df_test["passenger_count"] = pd.to_numeric(df_test["passenger_count"], errors="coerce")

df_test["pickup_longitude"] = df_test["pickup_longitude"].clip(-75.0, -72.0)
df_test["dropoff_longitude"] = df_test["dropoff_longitude"].clip(-75.0, -72.0)
df_test["pickup_latitude"] = df_test["pickup_latitude"].clip(40.0, 42.0)
df_test["dropoff_latitude"] = df_test["dropoff_latitude"].clip(40.0, 42.0)

df_test["passenger_count"] = (
    df_test["passenger_count"].fillna(1.0).clip(lower=1, upper=6)
)

df_test["distance"] = distance(
    df_test["pickup_latitude"],
    df_test["pickup_longitude"],
    df_test["dropoff_latitude"],
    df_test["dropoff_longitude"],
)

df_test["abs_lat_diff"] = (
    df_test["dropoff_latitude"] - df_test["pickup_latitude"]
).abs()
df_test["abs_lon_diff"] = (
    df_test["dropoff_longitude"] - df_test["pickup_longitude"]
).abs()

NYC_LAT, NYC_LON = 40.7141667, -74.0063889
df_test["pickup_dist_nyc"] = distance(
    df_test["pickup_latitude"], df_test["pickup_longitude"], NYC_LAT, NYC_LON
)
df_test["dropoff_dist_nyc"] = distance(
    df_test["dropoff_latitude"], df_test["dropoff_longitude"], NYC_LAT, NYC_LON
)

df_test["diff_long"] = (df_test.dropoff_longitude - df_test.pickup_longitude).abs()
df_test["diff_lat"] = (df_test.dropoff_latitude - df_test.pickup_latitude).abs()

df_test["distance"] = df_test["distance"].clip(lower=0.0, upper=35.0)
df_test["diff_long"] = df_test["diff_long"].clip(lower=0.0, upper=5.0)
df_test["diff_lat"] = df_test["diff_lat"].clip(lower=0.0, upper=5.0)

df_test["manhattan_dist"] = df_test["abs_lat_diff"] + df_test["abs_lon_diff"]
df_test["log_distance"] = np.log1p(df_test["distance"])

df_test["pickup_latitude"] = df_test["pickup_latitude"].astype(np.float32)
df_test["pickup_longitude"] = df_test["pickup_longitude"].astype(np.float32)
df_test["dropoff_latitude"] = df_test["dropoff_latitude"].astype(np.float32)
df_test["dropoff_longitude"] = df_test["dropoff_longitude"].astype(np.float32)

df_test["bearing"] = np.arctan2(
    (df_test["dropoff_longitude"] - df_test["pickup_longitude"]),
    (df_test["dropoff_latitude"] - df_test["pickup_latitude"]),
).astype(np.float32)
df_test["center_dist_change"] = (
    df_test["dropoff_dist_nyc"] - df_test["pickup_dist_nyc"]
).astype(np.float32)

df_test["pickup_grid_lat"] = np.floor(df_test["pickup_latitude"] * 100.0).astype(
    np.int32
)
df_test["pickup_grid_lon"] = np.floor(df_test["pickup_longitude"] * 100.0).astype(
    np.int32
)
df_test["dropoff_grid_lat"] = np.floor(df_test["dropoff_latitude"] * 100.0).astype(
    np.int32
)
df_test["dropoff_grid_lon"] = np.floor(df_test["dropoff_longitude"] * 100.0).astype(
    np.int32
)

X_kaggle_test = df_test[features].values



In [29]:
import xgboost as xgb

df_train_sorted = df_train.sort_values("pickup_datetime").reset_index(drop=True)
X_sorted = df_train_sorted[features].values
y_sorted = np.log1p(np.clip(df_train_sorted["fare_amount"].values, 0.0, None))

split_idx = int(0.7 * len(df_train_sorted))
X_train, X_valid = X_sorted[:split_idx], X_sorted[split_idx:]
y_train, y_valid = y_sorted[:split_idx], y_sorted[split_idx:]


def XGBmodel(x_train, x_valid, y_train, y_valid):
    matrix_train = xgb.DMatrix(x_train, label=y_train)
    matrix_valid = xgb.DMatrix(x_valid, label=y_valid)

    params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "seed": 0,
        "eta": 0.05,
        "max_depth": 8,
        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_lambda": 1.0,
        "reg_alpha": 0.0,
        "gamma": 0.0,
        "tree_method": "hist",
    }

    model = xgb.train(
        params=params,
        dtrain=matrix_train,
        num_boost_round=4000,
        early_stopping_rounds=50,
        evals=[(matrix_valid, "valid")],
        verbose_eval=False,
    )
    return model


# Change (score-relevant, minimal): keep the same early-stopping procedure to choose
# the best number of trees, then retrain on ALL cleaned data using that fixed number
# of rounds to improve fit for the final submission.
model_es = XGBmodel(X_train, X_valid, y_train, y_valid)
best_iter = getattr(model_es, "best_iteration", None)
best_rounds = int(best_iter + 1) if best_iter is not None else 4000

dall = xgb.DMatrix(X_sorted, label=y_sorted)
final_model = xgb.train(
    params={
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "seed": 0,
        "eta": 0.05,
        "max_depth": 8,
        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_lambda": 1.0,
        "reg_alpha": 0.0,
        "gamma": 0.0,
        "tree_method": "hist",
    },
    dtrain=dall,
    num_boost_round=best_rounds,
    evals=[],
    verbose_eval=False,
)

dtest = xgb.DMatrix(X_kaggle_test)
pred_log = final_model.predict(dtest)
prediction = np.expm1(pred_log)
prediction = np.clip(prediction, 0.0, None)



In [30]:
sample_sub = pd.read_csv(
    "../input/new-york-city-taxi-fare-prediction/sample_submission.csv"
)
fallback_fare = float(df_train["fare_amount"].median())

pred_series = pd.Series(prediction, index=df_test["key"])
submission = sample_sub.copy()
submission["fare_amount"] = submission["key"].map(pred_series).astype(float)
submission["fare_amount"] = (
    submission["fare_amount"].fillna(fallback_fare).clip(lower=0.0)
)

submission.to_csv("submission.csv", index=False)
submission.head()

,key,fare_amount
0,2010-10-01 21:26:11.0000001,6.842246
1,2013-10-06 01:38:00.00000083,30.251501
2,2012-03-30 19:13:53.0000001,4.105602
3,2012-02-08 02:57:23.0000001,29.726400
4,2013-12-13 22:56:00.000000237,31.255171
